# Serve reranker+embedding (GPU0) + Qwen3-4B-Instruct-2507 (GPU1) on Kaggle + tunnel

Kaggle GPU T4 x2 gives 2 physical GPUs. Pin each process to one via `CUDA_VISIBLE_DEVICES`:
- GPU0: FastAPI, embedding (dangvantuan/vietnamese-embedding, sentence-transformers) + reranker (AITeamVN/Vietnamese_Reranker, transformers), port 8001
- GPU1: vLLM OpenAI-compatible server, Qwen3-4B-Instruct-2507, port 8000

Setup on Kaggle before running: Settings -> Accelerator = GPU T4 x2, Internet = On.

Pick ONE tunnel section (ngrok or cloudflare), not both.

In [ ]:
!pip install -q -U vllm pyngrok fastapi uvicorn sentence-transformers

## GPU0: embedding + reranker server

In [ ]:
%%writefile rerank_embed_server.py
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

app = FastAPI()

embed_model = SentenceTransformer("dangvantuan/vietnamese-embedding", device="cuda")

rerank_tokenizer = AutoTokenizer.from_pretrained("AITeamVN/Vietnamese_Reranker")
rerank_model = (
    AutoModelForSequenceClassification.from_pretrained("AITeamVN/Vietnamese_Reranker")
    .to("cuda")
    .eval()
)


class EmbedReq(BaseModel):
    texts: list[str]


class RerankReq(BaseModel):
    query: str
    documents: list[str]


@app.post("/embed")
def embed(req: EmbedReq):
    vecs = embed_model.encode(req.texts)
    return {"embeddings": vecs.tolist()}


@app.post("/rerank")
def rerank(req: RerankReq):
    pairs = [[req.query, d] for d in req.documents]
    with torch.no_grad():
        inputs = rerank_tokenizer(
            pairs, padding=True, truncation=True, return_tensors="pt", max_length=2304
        ).to("cuda")
        scores = rerank_model(**inputs, return_dict=True).logits.view(-1).float().tolist()
    return {"scores": scores}

In [ ]:
import subprocess, time, requests, os

EMBED_PORT = 8001

embed_log = open("embed_rerank.log", "w")
embed_proc = subprocess.Popen(
    ["uvicorn", "rerank_embed_server:app", "--host", "0.0.0.0", "--port", str(EMBED_PORT)],
    stdout=embed_log, stderr=subprocess.STDOUT,
    env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"),
)

for _ in range(120):
    try:
        if requests.get(f"http://localhost:{EMBED_PORT}/docs", timeout=2).status_code == 200:
            print("Embedding/rerank server up.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    raise RuntimeError("Server did not start, check embed_rerank.log")


## GPU1: Qwen3-4B-Instruct-2507 (vLLM)

In [ ]:
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
LLM_PORT = 8000

server_log = open("vllm_server.log", "w")
server_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--port", str(LLM_PORT),
        "--dtype", "auto",  # T4 has no native bfloat16 support, let vllm pick
        "--max-model-len", "8192",
        "--gpu-memory-utilization", "0.90",
    ],
    stdout=server_log, stderr=subprocess.STDOUT,
    env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"),
)

for _ in range(180):  # up to ~15 min for model download + load
    if server_proc.poll() is not None:
        print(open("vllm_server.log").read()[-3000:])
        raise RuntimeError(f"vllm process exited early with code {server_proc.returncode}")
    try:
        if requests.get(f"http://localhost:{LLM_PORT}/health", timeout=2).status_code == 200:
            print("vLLM server up.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    print(open("vllm_server.log").read()[-3000:])
    raise RuntimeError("Server did not start, check vllm_server.log")


## Option A: ngrok tunnel (both ports)

In [ ]:
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = ""  # paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
assert NGROK_AUTHTOKEN, "set NGROK_AUTHTOKEN"
conf.get_default().auth_token = NGROK_AUTHTOKEN

llm_url = ngrok.connect(LLM_PORT, "http").public_url
embed_url = ngrok.connect(EMBED_PORT, "http").public_url
print("LLM base_url:", llm_url + "/v1")
print("Embed/rerank base_url:", embed_url)

## Option B: Cloudflare quick tunnel (no account needed, both ports)

In [ ]:
import subprocess, re, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


def start_tunnel(port, log_path):
    log = open(log_path, "w")
    proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=log, stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(30):
        time.sleep(2)
        text = open(log_path).read()
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", text)
        if m:
            url = m.group(0)
            break
    assert url, f"cloudflared url not found for port {port}, check {log_path}"
    return proc, url


llm_proc, llm_url = start_tunnel(LLM_PORT, "cloudflared_llm.log")
embed_proc, embed_url = start_tunnel(EMBED_PORT, "cloudflared_embed.log")
print("LLM base_url:", llm_url + "/v1")
print("Embed/rerank base_url:", embed_url)

## Call from local machine

Use whichever `llm_url` / `embed_url` printed above (ngrok or cloudflare).

LLM (OpenAI-compatible):
```python
from openai import OpenAI

client = OpenAI(base_url="<LLM_URL>/v1", api_key="none")
resp = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=[{"role": "user", "content": "Hello"}],
)
print(resp.choices[0].message.content)
```

Embedding + rerank:
```python
import requests

requests.post("<EMBED_URL>/embed", json={"texts": ["xin chao"]}).json()
requests.post("<EMBED_URL>/rerank", json={"query": "q", "documents": ["d1", "d2"]}).json()
```

Or curl:
```bash
curl <LLM_URL>/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "Qwen/Qwen3-4B-Instruct-2507", "messages": [{"role": "user", "content": "Hello"}]}'

curl <EMBED_URL>/embed -H "Content-Type: application/json" -d '{"texts": ["xin chao"]}'
```

In [ ]:
# keep notebook cell alive so Kaggle session + both servers + tunnels stay up
import time
while True:
    time.sleep(60)